In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], check=True)
import torch, ultralytics
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())
ultralytics.checks()

Ultralytics 8.4.190 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7108.1/8156.9 GB disk)


In [ ]:
import random, shutil, json, gc
from pathlib import Path
from PIL import Image, ImageDraw

SRC = next(p.parent for p in Path("/kaggle/input").rglob("Annotations")
           if (p.parent / "train.txt").exists() and (p.parent / "Images").exists())
print("SRC =", SRC)
DST = Path("/kaggle/working/widerperson_yolo")

KEEP, IGNORE = {1, 2, 3}, {4, 5}
MAX_PERSONS = 150          # v1: 80
MAX_IGNORE_AREA = 0.5      # drop only images where ignore regions cover more than half the image
MASK_COLOR = (114, 114, 114)
MASK_SKIP_OVERLAP = 0.3    # do not mask an ignore region if a labeled person has >= 30% of its area inside it
HOLDOUT, SEED = 500, 42    # same as v1 -> identical val / hold-out images

STAGE1 = dict(imgsz=640, epochs=20, batch=16)
STAGE2 = dict(imgsz=960, epochs=20, batch=8)

# Inference settings used by every evaluation call. Default = same protocol as v1 (NMS iou 0.6).
# Section 6b2 tunes this on val and updates it.
INFER = dict(iou=0.6)

# v1 results (640, from your earlier run) for the comparison table
V1 = dict(val=dict(precision=0.8274, recall=0.6977, mAP50=0.7918, mAP50_95=0.4954),
          test=dict(precision=0.8043, recall=0.6812, mAP50=0.7744, mAP50_95=0.4747))

assert (SRC / "Images").exists() and (SRC / "Annotations").exists(), "SRC path check karo"

In [ ]:
def read_ids(name):
    return [l.removesuffix(".jpg") for l in (SRC / f"{name}.txt").read_text().split()]


def read_ann(path):
    t = path.read_text().split()
    n = min(int(t[0]), (len(t) - 1) // 5)
    return [tuple(int(float(v)) for v in t[1 + 5 * k: 6 + 5 * k]) for k in range(n)]


def area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])


def inter(a, b):
    return area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))


def convert(img_id, split, is_train, stats):
    img_p = SRC / "Images" / f"{img_id}.jpg"
    ann_p = SRC / "Annotations" / f"{img_id}.jpg.txt"
    if not (img_p.exists() and ann_p.exists()):
        stats["missing"] += 1
        return
    with Image.open(img_p) as im:
        w, h = im.size
    persons, ignores = [], []
    for cls, x1, y1, x2, y2 in read_ann(ann_p):
        box = (max(0, x1), max(0, y1), min(w, x2), min(h, y2))
        if box[2] - box[0] <= 1 or box[3] - box[1] <= 1:
            continue
        if cls in KEEP:
            persons.append(box)
        elif cls in IGNORE:
            ignores.append(box)
    if is_train:
        if len(persons) > MAX_PERSONS:
            stats["dropped_dense"] += 1
            return
        if sum(area(b) for b in ignores) / (w * h) > MAX_IGNORE_AREA:
            stats["dropped_ignore"] += 1
            return
    rows = [f"0 {(b[0] + b[2]) / 2 / w:.6f} {(b[1] + b[3]) / 2 / h:.6f} {(b[2] - b[0]) / w:.6f} {(b[3] - b[1]) / h:.6f}"
            for b in persons]
    out_img = DST / "images" / split / img_p.name
    if is_train and ignores:
        im = Image.open(img_p).convert("RGB")
        draw = ImageDraw.Draw(im)
        for ib in ignores:
            if any(inter(pb, ib) >= MASK_SKIP_OVERLAP * area(pb) for pb in persons):
                stats["mask_skipped"] += 1
                continue
            draw.rectangle(ib, fill=MASK_COLOR)
            stats["masked_regions"] += 1
        im.save(out_img, quality=95)
    else:
        shutil.copy2(img_p, out_img)
    (DST / "labels" / split / f"{img_id}.txt").write_text("\n".join(rows) + ("\n" if rows else ""))
    stats["images"] += 1
    stats["boxes"] += len(rows)


if DST.exists():
    shutil.rmtree(DST)
for s in ["train", "val", "test"]:
    (DST / "images" / s).mkdir(parents=True)
    (DST / "labels" / s).mkdir(parents=True)

train_ids = read_ids("train")
random.Random(SEED).shuffle(train_ids)
splits = {"test": (train_ids[:HOLDOUT], False), "train": (train_ids[HOLDOUT:], True), "val": (read_ids("val"), False)}
for split, (ids, is_train) in splits.items():
    stats = dict(images=0, boxes=0, missing=0, dropped_dense=0, dropped_ignore=0, masked_regions=0, mask_skipped=0)
    for i in ids:
        convert(i, split, is_train, stats)
    print(f"{split:5s}", stats)

(DST / "data.yaml").write_text(
    f"path: {DST.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: person\n")
print((DST / "data.yaml").read_text())

In [ ]:
import matplotlib.pyplot as plt


def show(split="train", n=6):
    files = sorted((DST / "images" / split).glob("*.jpg"))
    imgs = random.Random(1).sample(files, min(n, len(files)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, p in zip(axes.ravel(), imgs):
        im = Image.open(p).convert("RGB")
        w, h = im.size
        d = ImageDraw.Draw(im)
        for line in (DST / "labels" / split / f"{p.stem}.txt").read_text().splitlines():
            _, cx, cy, bw, bh = map(float, line.split())
            d.rectangle([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h], outline="lime", width=2)
        ax.imshow(im); ax.set_title(p.name); ax.axis("off")
    plt.show()


show("train")

In [ ]:
from ultralytics import YOLO

COMMON = dict(data=str(DST / "data.yaml"), device=0, workers=4, optimizer="SGD", momentum=0.937,
              weight_decay=0.0005, cos_lr=True, mosaic=1.0, mixup=0.0, fliplr=0.5, scale=0.5, translate=0.1,
              degrees=0.0, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, cache="ram", amp=True, seed=0,
              project="/kaggle/working/runs", exist_ok=True, plots=True)

m1 = YOLO("yolov9m.pt")
m1.train(name="v2_stage1", lr0=0.005, lrf=0.01, warmup_epochs=3, close_mosaic=0, patience=25, **STAGE1, **COMMON)
RUN1 = Path(m1.trainer.save_dir)
BEST1 = RUN1 / "weights" / "best.pt"
print("Stage 1 best:", BEST1)
del m1
gc.collect(); torch.cuda.empty_cache()

In [ ]:
m2 = YOLO(str(BEST1))
m2.train(name="v2_stage2", lr0=0.002, lrf=0.05, warmup_epochs=1, close_mosaic=6, patience=20, **STAGE2, **COMMON)
RUN = Path(m2.trainer.save_dir)
WEIGHTS = RUN / "weights" / "best.pt"
print("Stage 2 best:", WEIGHTS)
del m2
gc.collect(); torch.cuda.empty_cache()
shutil.copy2(WEIGHTS, "/kaggle/working/best_v2.pt")

In [ ]:
import time
import numpy as np

IMGSZ, BATCH = STAGE2["imgsz"], STAGE2["batch"]
assert WEIGHTS.exists() and (DST / "data.yaml").exists()
model = YOLO(str(WEIGHTS))
print("Loaded", WEIGHTS)

In [ ]:
def run_val(data_yaml, split, name, plots=False):
    m = model.val(data=str(data_yaml), split=split, imgsz=IMGSZ, batch=BATCH, conf=0.001, **INFER,
                  plots=plots, project=str(RUN.parent), name=name, exist_ok=True, verbose=False)
    b = m.box
    return m, dict(precision=float(b.mp), recall=float(b.mr), mAP50=float(b.map50), mAP50_95=float(b.map))


mets, report = {}, {}
for split in ["val", "test"]:
    mets[split], report[split] = run_val(DST / "data.yaml", split, f"eval_{split}", plots=True)

print(f"{'split':8s}{'P':>8s}{'R':>8s}{'mAP50':>8s}{'mAP50-95':>10s}")
for s, r in report.items():
    print(f"{s:8s}{r['precision']:8.3f}{r['recall']:8.3f}{r['mAP50']:8.3f}{r['mAP50_95']:10.3f}")
gap = report["val"]["mAP50_95"] - report["test"]["mAP50_95"]
print(f"\nval - test gap in mAP50-95: {gap:+.3f}  (within ~0.02 = no meaningful selection bias)")

In [ ]:
print(f"{'':10s}{'split':6s}{'P':>8s}{'R':>8s}{'mAP50':>8s}{'mAP50-95':>10s}")
for s in ["val", "test"]:
    for tag, r in [("v1 (640)", V1[s]), (f"v2 ({IMGSZ})", report[s])]:
        print(f"{tag:10s}{s:6s}{r['precision']:8.3f}{r['recall']:8.3f}{r['mAP50']:8.3f}{r['mAP50_95']:10.3f}")
    d = {k: report[s][k] - V1[s][k] for k in V1[s]}
    print(f"{'delta':10s}{s:6s}" + "".join(f"{d[k]:+8.3f}" if k != 'mAP50_95' else f"{d[k]:+10.3f}" for k in
          ["precision", "recall", "mAP50", "mAP50_95"]))

print("\nInference size sweep on val (same weights):")
sweep_sizes = {}
for sz in [640, 800, 960, 1120]:
    m = model.val(data=str(DST / "data.yaml"), split="val", imgsz=sz, batch=BATCH, conf=0.001, **INFER,
                  plots=False, project=str(RUN.parent), name=f"sweep_{sz}", exist_ok=True, verbose=False)
    sweep_sizes[sz] = dict(recall=float(m.box.mr), mAP50=float(m.box.map50), mAP50_95=float(m.box.map))
    print(f"  imgsz {sz}: R={sweep_sizes[sz]['recall']:.3f}  mAP50={sweep_sizes[sz]['mAP50']:.3f}  "
          f"mAP50-95={sweep_sizes[sz]['mAP50_95']:.3f}")
report["imgsz_sweep_val"] = sweep_sizes

In [ ]:
nms = {}
for iou in [0.5, 0.6, 0.7, 0.8]:
    m = model.val(data=str(DST / "data.yaml"), split="val", imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=iou,
                  max_det=1000, plots=False, project=str(RUN.parent), name=f"nms_{int(iou * 100)}",
                  exist_ok=True, verbose=False)
    nms[iou] = dict(precision=float(m.box.mp), recall=float(m.box.mr), mAP50=float(m.box.map50),
                    mAP50_95=float(m.box.map))
    print(f"  NMS iou {iou}: P={nms[iou]['precision']:.3f} R={nms[iou]['recall']:.3f} "
          f"mAP50={nms[iou]['mAP50']:.3f} mAP50-95={nms[iou]['mAP50_95']:.3f}")
best_iou = max(nms, key=lambda k: nms[k]["mAP50_95"])
INFER = dict(iou=best_iou, max_det=1000)
print("\nChosen inference settings (best val mAP50-95):", INFER)
report["nms_sweep_val"] = {str(k): v for k, v in nms.items()}

try:
    m = model.val(data=str(DST / "data.yaml"), split="val", imgsz=IMGSZ, batch=BATCH, conf=0.001, augment=True,
                  plots=False, project=str(RUN.parent), name="tta_val", exist_ok=True, verbose=False, **INFER)
    report["tta_val"] = dict(recall=float(m.box.mr), mAP50=float(m.box.map50), mAP50_95=float(m.box.map))
    print("TTA on val (info only):", {k: round(v, 3) for k, v in report["tta_val"].items()})
except Exception as e:
    print("TTA check skipped:", repr(e))

tuned, mets_tuned = {}, {}
for split in ["val", "test"]:
    mets_tuned[split], tuned[split] = run_val(DST / "data.yaml", split, f"eval_{split}_tuned", plots=True)
print(f"\n{'':14s}{'split':6s}{'P':>8s}{'R':>8s}{'mAP50':>8s}{'mAP50-95':>10s}")
for s in ["val", "test"]:
    for tag, r in [("v2 official", report[s]), ("v2 tuned", tuned[s])]:
        print(f"{tag:14s}{s:6s}{r['precision']:8.3f}{r['recall']:8.3f}{r['mAP50']:8.3f}{r['mAP50_95']:10.3f}")
report["val_tuned"], report["test_tuned"] = tuned["val"], tuned["test"]
mets = mets_tuned      # confidence curves below now correspond to the tuned settings

In [ ]:
def curves(box):
    first = lambda a: np.array(a)[0]
    return np.array(box.px), first(box.p_curve), first(box.r_curve), first(box.f1_curve)


def conf_table(box, confs):
    px, P, R, F = curves(box)
    out = []
    for c in confs:
        i = int(np.abs(px - c).argmin())
        out.append((float(c), float(P[i]), float(R[i]), float(F[i])))
    return out


try:
    px, P, R, F = curves(mets["val"].box)
    Fs = np.convolve(F, np.ones(11) / 11, mode="same")
    BEST_CONF = float(px[int(Fs.argmax())])
    sweep = sorted(set([0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, round(BEST_CONF, 2)]))
    for s in ["val", "test"]:
        print(f"\n[{s}] conf    P      R      F1")
        for c, p, r, f in conf_table(mets[s].box, sweep):
            mark = "  <- chosen" if abs(c - round(BEST_CONF, 2)) < 1e-9 else ""
            print(f"       {c:4.2f}  {p:5.3f}  {r:5.3f}  {f:5.3f}{mark}")
    print(f"\nBEST_CONF (best F1 on val) = {BEST_CONF:.2f}")
except Exception as e:
    BEST_CONF = 0.25
    print("Curves not available, using conf 0.25. Reason:", repr(e))
print("Lower conf = higher recall, more false boxes. For crowd counting prefer a slightly lower value than the F1 optimum.")

In [ ]:
BINS = [("le10", 0, 10), ("11_30", 11, 30), ("31_80", 31, 80), ("gt80", 81, 10 ** 9), ("le80", 0, 80)]


def gt_count(lbl):
    return sum(1 for l in lbl.read_text().splitlines() if l.strip())


def make_subset(src_split, name, lo, hi):
    img_dir, lbl_dir = DST / "images" / name, DST / "labels" / name
    for d in (img_dir, lbl_dir):
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True)
    n = persons = 0
    for img in sorted((DST / "images" / src_split).glob("*.jpg")):
        lbl = DST / "labels" / src_split / f"{img.stem}.txt"
        if not lbl.exists():
            continue
        c = gt_count(lbl)
        if lo <= c <= hi:
            shutil.copy2(img, img_dir / img.name)
            shutil.copy2(lbl, lbl_dir / lbl.name)
            n += 1
            persons += c
    return n, persons


density, made = [], []
for split in ["val", "test"]:
    for tag, lo, hi in BINS:
        name = f"{split}_{tag}"
        n, persons = make_subset(split, name, lo, hi)
        made.append(name)
        if n < 30:
            print(f"skip {name}: only {n} images")
            continue
        y = DST / f"data_{name}.yaml"
        y.write_text(f"path: {DST.as_posix()}\ntrain: images/{name}\nval: images/{name}\nnames:\n  0: person\n")
        _, r = run_val(y, "val", f"eval_{name}")
        density.append(dict(split=split, bin=tag, images=n, persons=persons, **r))

print(f"{'split':6s}{'bin':8s}{'imgs':>6s}{'persons':>9s}{'P':>7s}{'R':>7s}{'mAP50':>8s}{'mAP50-95':>10s}")
for d in density:
    print(f"{d['split']:6s}{d['bin']:8s}{d['images']:6d}{d['persons']:9d}{d['precision']:7.3f}"
          f"{d['recall']:7.3f}{d['mAP50']:8.3f}{d['mAP50_95']:10.3f}")
report["density"] = density

In [ ]:
def predict_counts(split, conf):
    rows = []
    for r in model.predict(source=str(DST / "images" / split), stream=True, conf=conf, **INFER,
                           imgsz=IMGSZ, classes=[0], batch=BATCH, verbose=False):
        stem = Path(r.path).stem
        rows.append((Path(r.path).name, gt_count(DST / "labels" / split / f"{stem}.txt"), len(r.boxes)))
    return rows


def count_stats(rows):
    out = []
    for tag, lo, hi in BINS[:4] + [("all", 0, 10 ** 9)]:
        sel = [(g, p) for _, g, p in rows if lo <= g <= hi]
        if not sel:
            continue
        g, p = np.array(sel, dtype=float).T
        out.append(dict(bin=tag, images=len(sel), mean_gt=float(g.mean()), MAE=float(np.abs(p - g).mean()),
                        bias=float((p - g).mean()), rel_err=float(np.abs(p - g).sum() / max(g.sum(), 1))))
    return out


count_rows = {s: predict_counts(s, BEST_CONF) for s in ["test", "val"]}
counting = {}
for s, rows in count_rows.items():
    counting[s] = count_stats(rows)
    print(f"\n[{s}] conf={BEST_CONF:.2f}")
    print(f"{'bin':8s}{'imgs':>6s}{'mean GT':>9s}{'MAE':>8s}{'bias':>8s}{'rel.err':>9s}")
    for c in counting[s]:
        print(f"{c['bin']:8s}{c['images']:6d}{c['mean_gt']:9.1f}{c['MAE']:8.2f}{c['bias']:+8.2f}{c['rel_err']:9.1%}")
report["counting"] = counting

g = np.array([r[1] for r in count_rows["test"]])
p = np.array([r[2] for r in count_rows["test"]])
lim = max(g.max(), p.max()) * 1.05
plt.figure(figsize=(5.5, 5.5))
plt.scatter(g, p, s=10, alpha=0.5)
plt.plot([0, lim], [0, lim], "r--", lw=1)
plt.xlabel("labeled persons"); plt.ylabel("predicted persons"); plt.title("Hold-out test: count per image")
plt.xlim(0, lim); plt.ylim(0, lim)
plt.show()

In [ ]:
worst = sorted(count_rows["test"], key=lambda r: abs(r[2] - r[1]), reverse=True)[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (name, gt, pr) in zip(axes.ravel(), worst):
    res = model.predict(str(DST / "images" / "test" / name), conf=BEST_CONF, imgsz=IMGSZ, classes=[0], verbose=False, **INFER)[0]
    ax.imshow(res.plot(line_width=1, labels=False)[..., ::-1])
    ax.set_title(f"{name}  GT={gt}  pred={pr}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def bench(half, n=100):
    paths = sorted((DST / "images" / "test").glob("*.jpg"))[:n]
    model.predict(str(paths[0]), imgsz=IMGSZ, half=half, verbose=False)  # warm-up
    tot = []
    for pth in paths:
        r = model.predict(str(pth), imgsz=IMGSZ, conf=BEST_CONF, half=half, classes=[0], verbose=False, **INFER)[0]
        tot.append(sum(r.speed.values()))
    ms = float(np.mean(tot))
    return dict(ms_per_image=round(ms, 1), fps=round(1000 / ms, 1))


speed = {}
for half in [False, True]:
    try:
        speed["fp16" if half else "fp32"] = bench(half)
    except Exception as e:
        print("skip half =", half, repr(e))
print(speed)
report["speed_T4_batch1"] = speed

In [ ]:
report["recommended_conf"] = round(BEST_CONF, 2)
report["v1_reference"] = V1
report["config"] = dict(inference=INFER, stage1=STAGE1, stage2=STAGE2, max_persons=MAX_PERSONS, max_ignore_area=MAX_IGNORE_AREA,
                        ignore_regions="masked grey in train images", holdout=HOLDOUT, seed=SEED)
Path("/kaggle/working/eval_report.json").write_text(json.dumps(report, indent=2))

EXPORT = Path("/kaggle/working/export_v2")
shutil.rmtree(EXPORT, ignore_errors=True)
EXPORT.mkdir()
shutil.copy2(WEIGHTS, EXPORT / "best_v2.pt")
shutil.copy2("/kaggle/working/eval_report.json", EXPORT / "eval_report.json")
for name in ["results.csv", "results.png", "args.yaml", "confusion_matrix.png",
             "BoxPR_curve.png", "BoxF1_curve.png", "BoxP_curve.png", "BoxR_curve.png"]:
    if (RUN / name).exists():
        shutil.copy2(RUN / name, EXPORT / name)
for f in (RUN.parent / "eval_test").glob("Box*_curve.png"):
    shutil.copy2(f, EXPORT / f"holdout_{f.name}")

v, t = report["val"], report["test"]
vt, tt = report["val_tuned"], report["test_tuned"]
(EXPORT / "MODEL_CARD.md").write_text(f"""# YOLOv9m person detector v2

Two-stage fine-tune of COCO-pretrained YOLOv9m on WiderPerson (single class: person).
Stage 1: imgsz {STAGE1['imgsz']}, {STAGE1['epochs']} epochs. Stage 2: imgsz {STAGE2['imgsz']}, {STAGE2['epochs']} epochs.
Train set: <= {MAX_PERSONS} persons per image, ignore/crowd regions painted grey. Val and hold-out test are unfiltered.

| Model | Split | Precision | Recall | mAP50 | mAP50-95 |
|---|---|---|---|---|---|
| v1 (640) | val | {V1['val']['precision']:.3f} | {V1['val']['recall']:.3f} | {V1['val']['mAP50']:.3f} | {V1['val']['mAP50_95']:.3f} |
| v1 (640) | hold-out test | {V1['test']['precision']:.3f} | {V1['test']['recall']:.3f} | {V1['test']['mAP50']:.3f} | {V1['test']['mAP50_95']:.3f} |
| v2 ({IMGSZ}) | val | {v['precision']:.3f} | {v['recall']:.3f} | {v['mAP50']:.3f} | {v['mAP50_95']:.3f} |
| v2 ({IMGSZ}) | hold-out test | {t['precision']:.3f} | {t['recall']:.3f} | {t['mAP50']:.3f} | {t['mAP50_95']:.3f} |
| v2 ({IMGSZ}), tuned NMS | val | {vt['precision']:.3f} | {vt['recall']:.3f} | {vt['mAP50']:.3f} | {vt['mAP50_95']:.3f} |
| v2 ({IMGSZ}), tuned NMS | hold-out test | {tt['precision']:.3f} | {tt['recall']:.3f} | {tt['mAP50']:.3f} | {tt['mAP50_95']:.3f} |

Recommended inference settings: conf {BEST_CONF:.2f}, NMS iou {INFER['iou']}, max_det {INFER['max_det']}.
The v1 rows and the plain v2 rows use the same protocol (NMS iou 0.6); the tuned rows use the settings above.
""")
shutil.make_archive("/kaggle/working/yolov9m_person_v2_export", "zip", EXPORT)
print("Saved /kaggle/working/yolov9m_person_v2_export.zip and /kaggle/working/best_v2.pt")
print(sorted(p.name for p in EXPORT.iterdir()))

# Remove the converted dataset and temporary subsets so the committed Output stays small.
shutil.rmtree(DST, ignore_errors=True)
print("Converted dataset removed from /kaggle/working. Weights and reports are kept.")